In [1]:
#all necessary imports
import numpy as np
import pandas as pd
import polars as pl
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
import os

# TA indicators
import ta
from ta.trend import PSARIndicator

# Data Normalization and Scaling
from sklearn.preprocessing import MinMaxScaler
import numpy as np

# Scikit-learn utilities
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# hyperparameter optimization
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import f1_score
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Training device:", device)

print("Torch version:", torch.__version__)
print("Polars version:", pl.__version__)
print("TQDM imported successfully!")
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("TA library imported successfully!")
print("Optuna version:",optuna.__version__)

Training device: cuda
Torch version: 2.5.1+cu121
Polars version: 0.20.26
TQDM imported successfully!
NumPy version: 1.26.4
Pandas version: 2.2.2
TA library imported successfully!
Optuna version: 4.8.0


c:\Users\User\miniconda3\envs\quant_dl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# NASDAQ DATA

# 
symbols = [
    "GOOGL","AVGO","MSFT","META","NVDA","AAPL","TSLA",
]

dfs = {}

for s in symbols:
    filepath = f"NASDAQ_DATA/{s}_daily.csv"
    
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue
    
    df = pd.read_csv(filepath)
    dfs[s] = df
    print(f"\n── {s} ──────────────────────────")
    print(f"  Shape : {df.shape}")
    print(f"  Head  :\n{df.head()}")
    print(f"  Dtypes:\n{df.dtypes}")
    print(f"  Nulls :\n{df.isnull().sum()}")

print(f"\n── Loaded {len(dfs)}/{len(symbols)} tickers ──")


── GOOGL ──────────────────────────
  Shape : (5282, 62)
  Head  :
              datetime ticker  open_price  high_price  low_price  close_price  \
0  2005-06-03 21:30:00  GOOGL    7.176706    7.239517   6.941975     7.013297   
1  2005-06-06 21:30:00  GOOGL    7.065850    7.350873   7.052586     7.280554   
2  2005-06-07 21:30:00  GOOGL    7.444714    7.497017   7.264541     7.335111   
3  2005-06-08 21:30:00  GOOGL    7.328354    7.336858   6.956742     6.995781   
4  2005-06-09 21:30:00  GOOGL    7.124904    7.219496   7.020805     7.164695   

         volume    EMA_10    EMA_20  EMA_ratio  ...  volume_lag_3  \
0  7.519087e+08  6.722983  6.354545   1.103666  ...  8.902334e+08   
1  9.031041e+08  6.824359  6.442736   1.130041  ...  1.408767e+09   
2  9.726738e+08  6.917223  6.527724   1.123686  ...  7.186961e+08   
3  1.031882e+09  6.931506  6.572301   1.064434  ...  7.519087e+08   
4  6.576569e+08  6.973904  6.628719   1.080857  ...  9.031041e+08   

   volume_lag_4  volume_lag_5 

In [3]:
for symbol in symbols:
    df = pd.read_csv(f'NASDAQ_DATA/{symbol}_daily.csv', parse_dates=["datetime"])
    
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) * 100 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)
    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up
    df["volume_log"]            = np.log1p(df["volume"])
    df["volume_pct_change_log"] = df["volume_log"].diff(1).shift(-1)
    df = df.dropna().reset_index(drop=True)

    for col, sigma in {'price_pct_change': 3, 'volume_pct_change_log': 2}.items():
        mean, std = df[col].mean(), df[col].std()
        df = df[df[col].between(mean - sigma * std, mean + sigma * std)]
    df = df.reset_index(drop=True)

    # Date-based split: train = up to 2018-12-31, test = 2019 onwards
    train_df = df[df['datetime'] <= '2018-12-31']
    test_df  = df[df['datetime'] >  '2018-12-31']

    target_scaler = StandardScaler()
    train_y = target_scaler.fit_transform(train_df[['price_pct_change', 'volume_pct_change_log']])

    print(f"\n{symbol}")
    print(f"  Raw train UP%      : {(train_df['price_pct_change'] > 0).mean():.2%}")
    print(f"  Scaled train mean  : {train_y[:, 0].mean():.4f}")
    print(f"  Scaled train std   : {train_y[:, 0].std():.4f}")
    print(f"  Train date range   : {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Test  date range   : {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")
    print(f"  Train rows: {len(train_df)}  Test rows: {len(test_df)}")


GOOGL
  Raw train UP%      : 51.45%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2005-06-13 → 2018-12-28
  Test  date range   : 2018-12-31 → 2026-06-01
  Train rows: 3170  Test rows: 1750

AVGO
  Raw train UP%      : 53.23%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.0000
  Train date range   : 2010-06-01 → 2018-12-28
  Test  date range   : 2018-12-31 → 2026-06-01
  Train rows: 1999  Test rows: 1771

MSFT
  Raw train UP%      : 49.74%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Train date range   : 1999-03-12 → 2018-12-28
  Test  date range   : 2018-12-31 → 2026-06-01
  Train rows: 4606  Test rows: 1770

META
  Raw train UP%      : 53.20%
  Scaled train mean  : -0.0000
  Scaled train std   : 1.0000
  Train date range   : 2013-03-15 → 2018-12-28
  Test  date range   : 2018-12-31 → 2026-06-01
  Train rows: 1361  Test rows: 1728

NVDA
  Raw train UP%      : 50.57%
  Scaled train mean  : 0.0000
  Scaled train std   : 1.00

In [4]:
import pickle
import os

SEQ_LEN_DEFAULT = 30
prepared = {}
scalers  = {}

FEATURE_COLS = [
    # stationary price action (replaces raw price/volume levels)
    "ret_1", "gap", "intraday_range", "body",
    # trend (stationary)
    "EMA_ratio", "macd_norm",
    # momentum
    "RSI_14", "ROC_5", "ROC_10", "ROC_20",
    # volatility (ratios, not levels)
    "ATR_ratio", "BB_pct", "realized_vol",
    # volume (stationary)
    "OBV_momentum", "volume_ratio", "volume_surge", "MFI_14",
    # lagged returns
    "returns_lag_1", "returns_lag_2", "returns_lag_3", "returns_lag_4", "returns_lag_5",
    # regime / persistence
    "above_ema10", "above_ema20", "roc5_pos", "roc20_pos", "up_streak",
]

TARGET_COLS = ["price_pct_change_5d_vn"]


def _make_seqs(X, y, seq_len):
    """Simple sliding-window builder."""
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        ws.append(1.0)
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


for symbol in symbols:
    filepath = f"NASDAQ_DATA/{symbol}_daily.csv"
    if not os.path.exists(filepath):
        print(f"  WARNING — {filepath} not found, skipping.")
        continue

    df = pd.read_csv(filepath, parse_dates=["datetime"])

    # ── Lag features ──────────────────────────────────────────────────────────
    for lag in range(1, 6):
        df[f"close_lag_{lag}"]   = df["close_price"].shift(lag)
        df[f"returns_lag_{lag}"] = df["close_price"].pct_change(1).shift(lag) 
        df[f"volume_lag_{lag}"]  = df["volume"].shift(lag)

    # ── Regime / persistence features ─────────────────────────────────────────
    df["above_ema10"] = (df["close_price"] > df["EMA_10"]).astype(float)
    df["above_ema20"] = (df["close_price"] > df["EMA_20"]).astype(float)
    df["roc5_pos"]    = (df["ROC_5"] > 0).astype(float)
    df["roc20_pos"]   = (df["ROC_20"] > 0).astype(float)

    daily_up  = df["close_price"].diff(1).gt(0).astype(int)
    streak_id = (daily_up != daily_up.shift()).cumsum()
    df["up_streak"] = (daily_up.groupby(streak_id).cumcount() + 1) * daily_up

    # ── Stationary price-action features (regime-transferable) ────────────────
    df["ret_1"]          = df["close_price"].pct_change(1)
    df["gap"]            = df["open_price"] / df["close_price"].shift(1) - 1
    df["intraday_range"] = (df["high_price"] - df["low_price"]) / df["close_price"]
    df["body"]           = (df["close_price"] - df["open_price"]) / df["open_price"]
    df["macd_norm"]      = df["MACD_hist"] / df["close_price"]

    # ── 5-day forward close % change target ───────────────────────────────────
    df["price_pct_change_5d"] = (df["close_price"].shift(-5) / df["close_price"] - 1) * 100
    # trailing daily-return vol (no look-ahead), scaled to 5d horizon, in %, floored
    _daily_ret = df["close_price"].pct_change()
    df["vol_denom_5d"] = (_daily_ret.rolling(20).std() * np.sqrt(5) * 100).clip(lower=0.5)
    # target = 5d return in units of its own trailing 5d volatility (z-score)
    df["price_pct_change_5d_vn"] = df["price_pct_change_5d"] / df["vol_denom_5d"]

    df = df.dropna().reset_index(drop=True)

    # ── (Outlier clipping moved BELOW the split: TRAIN stats only, no test leakage) ──


    # ── 3-way split (no leakage): val carved from pre-2019; test = 2019+ touched once ──
    pre  = df[df['datetime'] <  '2019-01-01'].reset_index(drop=True)   # 2018-12-31 bar stays OUT of test
    post = df[df['datetime'] >= '2019-01-01']                        # test strictly 2019+
    cut  = int(len(pre) * 0.85)
    train_df = pre.iloc[:cut - 5]    # 5-row embargo: last train 5d targets must not reach val
    val_df   = pre.iloc[cut:-5]      # 5-row embargo: last val 5d targets must not reach test (2019)
    test_df  = post

    # ── Clip outliers on TRAIN ONLY (band from train); val/test left as-is (no leakage) ──
    for col, sigma in {"price_pct_change_5d_vn": 3}.items():
        mean, std = train_df[col].mean(), train_df[col].std()
        train_df  = train_df[train_df[col].between(mean - sigma * std, mean + sigma * std)]

    train_X_raw = train_df[FEATURE_COLS].values.astype(np.float32)
    val_X_raw   = val_df[FEATURE_COLS].values.astype(np.float32)
    test_X_raw  = test_df[FEATURE_COLS].values.astype(np.float32)

    train_y_raw = train_df[TARGET_COLS].values.astype(np.float32)
    val_y_raw   = val_df[TARGET_COLS].values.astype(np.float32)
    test_y_raw  = test_df[TARGET_COLS].values.astype(np.float32)

    # ── Fit scalers on train only ─────────────────────────────────────────────
    feature_scaler = StandardScaler()
    target_scaler  = StandardScaler()

    train_X_scaled = feature_scaler.fit_transform(train_X_raw)
    val_X_scaled   = feature_scaler.transform(val_X_raw)
    test_X_scaled  = feature_scaler.transform(test_X_raw)

    train_y_scaled = target_scaler.fit_transform(train_y_raw)
    val_y_scaled   = target_scaler.transform(val_y_raw)
    test_y_scaled  = target_scaler.transform(test_y_raw)

    # ── Save scalers ──────────────────────────────────────────────────────────
    os.makedirs(f"models/{symbol}", exist_ok=True)
    with open(f"models/{symbol}/{symbol}_feature_scaler.pkl", "wb") as f:
        pickle.dump(feature_scaler, f)
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "wb") as f:
        pickle.dump(target_scaler, f)

    # ── Pre-build sequences at default SEQ_LEN ────────────────────────────────
    X_train, y_train, _ = _make_seqs(train_X_scaled, train_y_scaled, SEQ_LEN_DEFAULT)
    X_test,  y_test,  _ = _make_seqs(test_X_scaled,  test_y_scaled,  SEQ_LEN_DEFAULT)

    prepared[symbol] = {
        "train_X_scaled": train_X_scaled,
        "train_y_scaled": train_y_scaled,
        "test_X_scaled":  test_X_scaled,
        "test_y_scaled":  test_y_scaled,
        "val_X_scaled":   val_X_scaled,
        "val_y_scaled":   val_y_scaled,
        "test_vol_denom": test_df["vol_denom_5d"].values.astype(np.float32),
        "X_train": X_train,  "y_train": y_train,
        "X_test":  X_test,   "y_test":  y_test,
    }
    scalers[symbol] = {
        "feature_scaler": feature_scaler,
        "target_scaler":  target_scaler,
    }

    print(f"\n── {symbol} {'─' * 60}")
    print(f"  Rows  → train:{len(train_df)}  val:{len(val_df)}  test:{len(test_df)}")
    print(f"  Seqs  → train:{X_train.shape[0]}  test:{X_test.shape[0]}")
    print(f"  Shape → X:{X_train.shape}  y:{y_train.shape}")
    print(f"  Train date range: {train_df['datetime'].min().date()} → {train_df['datetime'].max().date()}")
    print(f"  Test  date range: {test_df['datetime'].min().date()} → {test_df['datetime'].max().date()}")
    print(f"  Scalers saved → models/{symbol}/")

print(f"\n── prepared built for {len(prepared)} symbols ──")


── GOOGL ────────────────────────────────────────────────────────────
  Rows  → train:2838  val:505  test:1859
  Seqs  → train:2808  test:1829
  Shape → X:(2808, 30, 27)  y:(2808, 1)
  Train date range: 2005-07-01 → 2016-12-12
  Test  date range: 2019-01-02 → 2026-05-26
  Scalers saved → models/GOOGL/

── AVGO ────────────────────────────────────────────────────────────
  Rows  → train:1799  val:318  test:1859
  Seqs  → train:1769  test:1829
  Shape → X:(1769, 30, 27)  y:(1769, 1)
  Train date range: 2010-06-21 → 2017-09-11
  Test  date range: 2019-01-02 → 2026-05-26
  Scalers saved → models/AVGO/

── MSFT ────────────────────────────────────────────────────────────
  Rows  → train:4172  val:743  test:1859
  Seqs  → train:4142  test:1829
  Shape → X:(4142, 30, 27)  y:(4142, 1)
  Train date range: 1999-03-12 → 2016-01-04
  Test  date range: 2019-01-02 → 2026-05-26
  Scalers saved → models/MSFT/

── META ────────────────────────────────────────────────────────────
  Rows  → train:1214  

In [5]:
import torch
from torch.utils.data import Dataset, DataLoader
import numpy as np


# ── Dataset ───────────────────────────────────────────────────────────────────
class TadawulDataset(Dataset):
    def __init__(self, X, y, weights=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)
        self.w = torch.tensor(weights, dtype=torch.float32) \
                 if weights is not None \
                 else torch.ones(len(X), dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.w[idx]


BATCH_SIZE = 32
loaders    = {}

for symbol, data in prepared.items():
    train_loader = DataLoader(TadawulDataset(data["X_train"], data["y_train"]), batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(TadawulDataset(data["X_test"],  data["y_test"]),  batch_size=BATCH_SIZE, shuffle=False)

    loaders[symbol] = {
        "train": train_loader,
        "test":  test_loader,
    }

    print(f"\n{symbol} — Train: {len(train_loader)} batches | Test: {len(test_loader)} batches")

    sample_X, sample_y, sample_w = next(iter(train_loader))
    print(f"  Sample X: {sample_X.shape} → (batch, seq_len, features)")
    print(f"  Sample y: {sample_y.shape} → (batch, 1)")
    print(f"  Sample w: {sample_w.shape} → (batch,)")


GOOGL — Train: 88 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 27]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

AVGO — Train: 56 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 27]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

MSFT — Train: 130 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 27]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

META — Train: 37 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 27]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

NVDA — Train: 125 batches | Test: 58 batches
  Sample X: torch.Size([32, 30, 27]) → (batch, seq_len, features)
  Sample y: torch.Size([32, 1]) → (batch, 1)
  Sample w: torch.Size([32]) → (batch,)

AAPL — Train: 13

In [6]:
import optuna
import torch
import torch.nn as nn
import numpy as np
from torch.utils.data import DataLoader, WeightedRandomSampler
from sklearn.metrics import balanced_accuracy_score
import torch.nn.functional as F
import json
import os

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

best_params_all = {}

# ── BiLSTM + Attention Model (with LayerNorm) ─────────────────────────────────
class StockPctChangeBiLSTMAttention(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout=0.2):
        super().__init__()
        self.lstm      = nn.LSTM(input_size, hidden_size, num_layers,
                                 batch_first=True, bidirectional=True,
                                 dropout=dropout if num_layers > 1 else 0)
        self.attention      = nn.Linear(hidden_size * 2, 1)
        self.pos_bias       = nn.Parameter(torch.zeros(1))
        self.ln             = nn.LayerNorm(hidden_size * 2)
        self.dropout        = nn.Dropout(dropout)
        self.fc             = nn.Linear(hidden_size * 2, 1)
        self.skip_fc        = nn.Linear(hidden_size * 2, 1)
        self.output_blend   = nn.Parameter(torch.tensor(0.5))

    def forward(self, x):
        lstm_out, _  = self.lstm(x)
        seq_len      = lstm_out.size(1)
        pos_weights  = torch.linspace(0, 1, seq_len, device=x.device).unsqueeze(-1)
        attn_raw     = self.attention(lstm_out) + self.pos_bias * pos_weights
        attn_weights = torch.softmax(attn_raw, dim=1)
        context      = (attn_weights * lstm_out).sum(dim=1)
        out_attn     = self.fc(self.dropout(self.ln(context)))
        last_hidden  = lstm_out[:, -1, :]
        out_skip     = self.skip_fc(last_hidden)
        alpha        = torch.sigmoid(self.output_blend)
        return alpha * out_attn + (1 - alpha) * out_skip


# ── Joint Loss ────────────────────────────────────────────────────────────────
class JointPredictionLoss(nn.Module):
    def __init__(self, alpha=2.0, beta=0.5, gamma=1.0):
        super().__init__()
        self.alpha = alpha
        self.beta  = beta
        self.gamma = gamma

    def forward(self, preds, targets, weights=None):
        # Plain SmoothL1 (Huber, delta=1.0) on the standardized target.
        # Anti-collapse stack removed (BCE / std_match / collapse / sign):
        # it drove the mean-drift collapse. alpha/beta/gamma/weights kept in
        # the signature only for call-site compatibility; unused.
        return nn.SmoothL1Loss(beta=1.0)(preds, targets)


# ── Helpers ───────────────────────────────────────────────────────────────────
def create_sequences(X, y, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i : i + seq_len])
        ys.append(y[i + seq_len])
        move   = abs(y[i + seq_len, 0])
        ws.append(1.0 + 2.0 * float(move > move_threshold))
    return np.array(Xs), np.array(ys), np.array(ws, dtype=np.float32)


def make_balanced_sampler(y, oversample_factor=2.0):
    labels = (y[:, 0] > 0).astype(int)
    moves  = np.abs(y[:, 0])
    median_move = np.median(moves)
    bucket = labels * 2 + (moves > median_move).astype(int)
    counts = np.bincount(bucket, minlength=4).astype(float)
    counts = np.maximum(counts, 1)
    weight_map = 1.0 / counts
    weight_map[1] *= oversample_factor
    weight_map[3] *= oversample_factor
    sample_weights = torch.tensor(weight_map[bucket], dtype=torch.float32)
    return WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)


def safe_corr(a, b):
    if np.std(a) < 1e-8 or np.std(b) < 1e-8:
        return 0.0
    r = np.corrcoef(a, b)[0, 1]
    return 0.0 if np.isnan(r) else float(r)


# ── Clean NaN values ──────────────────────────────────────────────────────────
print("\nCleaning NaN values from all symbols...")
for symbol, data in prepared.items():
    for split in ("train", "val", "test"):
        X_key, y_key = f"{split}_X_scaled", f"{split}_y_scaled"
        X, y         = data[X_key], data[y_key]
        mask         = ~(np.isnan(X).any(axis=1) | np.isnan(y).any(axis=1))
        before       = len(X)
        data[X_key]  = X[mask]
        data[y_key]  = y[mask]
        if split == "test" and "test_vol_denom" in data:
            data["test_vol_denom"] = data["test_vol_denom"][mask]
        print(f"  {symbol} {split}: {before} → {len(data[X_key])} rows")


# ── Pooled-dataset helpers (v3: ONE model across all symbols) ─────────────────
# Sequences are built PER SYMBOL (windows never cross a symbol boundary), then a
# symbol one-hot is appended to every timestep and the symbols are concatenated.
POOL_SYMS = [s for s in symbols if s in prepared]
SYM2IDX   = {s: i for i, s in enumerate(POOL_SYMS)}
N_SYM     = len(POOL_SYMS)
POOLED_INPUT_SIZE = next(iter(prepared.values()))["train_X_scaled"].shape[1] + N_SYM

def _append_onehot(Xseq, symbol):
    oh = np.zeros((len(Xseq), Xseq.shape[1], N_SYM), dtype=np.float32)
    oh[:, :, SYM2IDX[symbol]] = 1.0
    return np.concatenate([Xseq, oh], axis=2)

def build_pooled(split, seq_len, move_threshold=0.3):
    Xs, ys, ws = [], [], []
    for s in POOL_SYMS:
        d = prepared[s]
        Xseq, yseq, wseq = create_sequences(d[f"{split}_X_scaled"], d[f"{split}_y_scaled"], seq_len, move_threshold)
        if len(Xseq) == 0:
            continue
        Xs.append(_append_onehot(Xseq, s)); ys.append(yseq); ws.append(wseq)
    return np.concatenate(Xs), np.concatenate(ys), np.concatenate(ws)

def build_symbol(symbol, split, seq_len, move_threshold=0.3):
    d = prepared[symbol]
    Xseq, yseq, wseq = create_sequences(d[f"{split}_X_scaled"], d[f"{split}_y_scaled"], seq_len, move_threshold)
    return _append_onehot(Xseq, symbol), yseq, wseq

print(f"Pooled: {N_SYM} symbols {POOL_SYMS} -> input_size {POOLED_INPUT_SIZE} (feats + {N_SYM} one-hot)")


Using device: cuda

Cleaning NaN values from all symbols...
  GOOGL train: 2838 → 2838 rows
  GOOGL val: 505 → 505 rows
  GOOGL test: 1859 → 1859 rows
  AVGO train: 1799 → 1799 rows
  AVGO val: 318 → 318 rows
  AVGO test: 1859 → 1859 rows
  MSFT train: 4172 → 4172 rows
  MSFT val: 743 → 743 rows
  MSFT test: 1859 → 1859 rows
  META train: 1214 → 1214 rows
  META val: 212 → 212 rows
  META test: 1859 → 1859 rows
  NVDA train: 4019 → 4019 rows
  NVDA val: 715 → 715 rows
  NVDA test: 1859 → 1859 rows
  AAPL train: 4184 → 4184 rows
  AAPL val: 743 → 743 rows
  AAPL test: 1859 → 1859 rows
  TSLA train: 1610 → 1610 rows
  TSLA val: 284 → 284 rows
  TSLA test: 1859 → 1859 rows
Pooled: 7 symbols ['GOOGL', 'AVGO', 'MSFT', 'META', 'NVDA', 'AAPL', 'TSLA'] -> input_size 34 (feats + 7 one-hot)


In [7]:
# ── Optuna search (POOLED: single model over all symbols) ─────────────────────
import os, json

input_size = POOLED_INPUT_SIZE

def objective(trial):
    hidden_size    = trial.suggest_categorical("hidden_size", [64, 128])
    num_layers     = trial.suggest_int("num_layers", 2, 3)
    dropout        = trial.suggest_float("dropout", 0.2, 0.4, step=0.1)
    lr             = trial.suggest_float("lr", 1e-4, 5e-3, log=True)
    batch_size     = trial.suggest_categorical("batch_size", [64, 128])
    seq_len        = trial.suggest_categorical("seq_len", [10, 20, 30, 40])
    move_threshold = trial.suggest_float("move_threshold", 0.5, 2.0, step=0.1)

    X_tr, y_tr, w_tr = build_pooled("train", seq_len, move_threshold)
    X_vl, y_vl, _    = build_pooled("val",   seq_len, move_threshold)

    train_ld = DataLoader(TadawulDataset(X_tr, y_tr, w_tr), batch_size=batch_size, shuffle=True, drop_last=True)
    val_ld   = DataLoader(TadawulDataset(X_vl, y_vl), batch_size=batch_size, shuffle=False)

    torch.manual_seed(42); np.random.seed(42)
    model = StockPctChangeBiLSTMAttention(input_size=input_size, hidden_size=hidden_size,
                                          num_layers=num_layers, dropout=dropout).to(device)
    criterion = JointPredictionLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    EPOCHS, PATIENCE, best_score, pc = 40, 8, -1e9, 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        for xb, yb, wb in train_ld:
            optimizer.zero_grad()
            loss = criterion(model(xb.to(device)), yb.to(device), wb.to(device))
            loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()

        model.eval(); P, A = [], []
        with torch.no_grad():
            for xb, yb, _ in val_ld:
                P.append(model(xb.to(device)).cpu().numpy()); A.append(yb.numpy())
        P, A = np.vstack(P), np.vstack(A)
        dir_ = balanced_accuracy_score(np.sign(A[:, 0]).astype(int), np.sign(P[:, 0]).astype(int))
        corr = safe_corr(A[:, 0], P[:, 0])
        std_ratio = np.std(P[:, 0]) / max(np.std(A[:, 0]), 1e-6)
        score = 0.5 * dir_ + 0.5 * max(corr, 0.0) - abs(1.0 - std_ratio) * 0.2

        if score > best_score: best_score, pc = score, 0
        else:
            pc += 1
            if pc >= PATIENCE: break
        trial.report(score, epoch)
        if trial.should_prune(): raise optuna.exceptions.TrialPruned()
    return best_score

study = optuna.create_study(direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=5),
    study_name="bilstm_pooled")
study.optimize(objective, n_trials=30, timeout=7200)

best_params_pooled = study.best_params
os.makedirs("models/pooled_5d", exist_ok=True)
with open("models/pooled_5d/pooled_best_params.json", "w") as f:
    json.dump({"best_params": study.best_params, "best_value": float(study.best_value)}, f, indent=2)
print("Best pooled value:", round(float(study.best_value), 4))
print("Best pooled params:", study.best_params)
print("Saved -> models/pooled_5d/pooled_best_params.json")


[I 2026-06-24 01:48:21,986] A new study created in memory with name: bilstm_pooled
[I 2026-06-24 01:49:50,776] Trial 0 finished with value: 0.19896783465604007 and parameters: {'hidden_size': 128, 'num_layers': 3, 'dropout': 0.30000000000000004, 'lr': 0.00018410729205738696, 'batch_size': 64, 'seq_len': 10, 'move_threshold': 2.0}. Best is trial 0 with value: 0.19896783465604007.
[I 2026-06-24 01:51:16,174] Trial 1 finished with value: 0.21692513342667602 and parameters: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2, 'lr': 0.0003287747413991121, 'batch_size': 64, 'seq_len': 20, 'move_threshold': 1.0}. Best is trial 1 with value: 0.21692513342667602.
[I 2026-06-24 01:52:20,558] Trial 2 finished with value: 0.20973247305626253 and parameters: {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.30000000000000004, 'lr': 0.0010150667045928574, 'batch_size': 128, 'seq_len': 40, 'move_threshold': 1.7000000000000002}. Best is trial 1 with value: 0.21692513342667602.
[I 2026-06-24 01:54:05,

Best pooled value: 0.2518
Best pooled params: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0048530602352941194, 'batch_size': 64, 'seq_len': 10, 'move_threshold': 1.0}
Saved -> models/pooled_5d/pooled_best_params.json


In [8]:
# Training (POOLED: ONE model across all symbols)
import torch, torch.nn as nn, numpy as np, os, json, time
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.metrics import balanced_accuracy_score

try:
    best = best_params_pooled
except NameError:
    with open("models/pooled_5d/pooled_best_params.json") as f:
        best = json.load(f)["best_params"]
    print("Loaded pooled params from JSON")
print("Pooled params:", best)

mt = best.get("move_threshold", 1.0)
X_tr, y_tr, w_tr = build_pooled("train", best["seq_len"], mt)
X_vl, y_vl, _    = build_pooled("val",   best["seq_len"], mt)
print("Pooled train seqs:", len(X_tr), " val seqs:", len(X_vl), " input_size:", X_tr.shape[2])

train_loader = DataLoader(TadawulDataset(X_tr, y_tr, w_tr), batch_size=best["batch_size"], shuffle=True, drop_last=True)
val_loader   = DataLoader(TadawulDataset(X_vl, y_vl), batch_size=best["batch_size"], shuffle=False)

torch.manual_seed(42); np.random.seed(42)
model = StockPctChangeBiLSTMAttention(input_size=POOLED_INPUT_SIZE, hidden_size=best["hidden_size"],
                                      num_layers=best["num_layers"], dropout=best["dropout"]).to(device)
criterion = JointPredictionLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=best["lr"], weight_decay=1e-5)
scheduler = ReduceLROnPlateau(optimizer, mode="max", patience=5, factor=0.5)

os.makedirs("models/pooled_5d", exist_ok=True)
save_path = "models/pooled_5d/pooled_best_bilstm.pt"

EPOCHS, PATIENCE, best_score, pc = 100, 15, -1e9, 0
print(f"{'Ep':>4} {'TrLoss':>9} {'ValDir':>7} {'ValCorr':>8} {'UP%':>5} {'StdR':>5}")
for epoch in range(1, EPOCHS + 1):
    model.train(); tl = []
    for xb, yb, wb in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(xb.to(device)), yb.to(device), wb.to(device))
        loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
        tl.append(loss.item())
    model.eval(); P, A = [], []
    with torch.no_grad():
        for xb, yb, _ in val_loader:
            P.append(model(xb.to(device)).cpu().numpy()); A.append(yb.numpy())
    P, A = np.vstack(P), np.vstack(A)
    vdir  = balanced_accuracy_score(np.sign(A[:, 0]).astype(int), np.sign(P[:, 0]).astype(int))
    vcorr = safe_corr(A[:, 0], P[:, 0])
    up    = (P[:, 0] > 0).mean()
    sr    = np.std(P[:, 0]) / max(np.std(A[:, 0]), 1e-6)
    score = 0.5 * vdir + 0.5 * max(vcorr, 0.0) - abs(1.0 - sr) * 0.2
    print(f"{epoch:>4} {np.mean(tl):>9.5f} {vdir:>7.3f} {vcorr:>8.3f} {up:>4.0%} {sr:>5.2f}")
    scheduler.step(score)
    if score > best_score:
        best_score, pc = score, 0
        torch.save(model.state_dict(), save_path); print("       saved")
    else:
        pc += 1
        if pc >= PATIENCE:
            print("Early stop @ epoch", epoch); break
print("Best pooled val score:", round(best_score, 4), "->", save_path)
trained_models = {"pooled": {"best_score": best_score}}


Pooled params: {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.4, 'lr': 0.0048530602352941194, 'batch_size': 64, 'seq_len': 10, 'move_threshold': 1.0}
Pooled train seqs: 19766  val seqs: 3450  input_size: 34
  Ep    TrLoss  ValDir  ValCorr   UP%  StdR
   1   0.42048   0.509    0.106  91%  0.08
       saved
   2   0.41116   0.493    0.002  79%  0.10
   3   0.40573   0.518    0.092  43%  0.12
       saved
   4   0.39809   0.515    0.042  50%  0.14
   5   0.37710   0.527    0.092  58%  0.28
       saved
   6   0.34698   0.538    0.113  47%  0.30
       saved
   7   0.30223   0.531    0.091  49%  0.42
       saved
   8   0.25836   0.543    0.101  47%  0.54
       saved
   9   0.22803   0.546    0.125  48%  0.49
       saved
  10   0.20041   0.543    0.106  43%  0.53
  11   0.17891   0.545    0.124  45%  0.58
       saved
  12   0.16208   0.527    0.084  49%  0.50
  13   0.15132   0.549    0.113  47%  0.59
  14   0.13956   0.548    0.121  40%  0.49
  15   0.13066   0.528    0.079  49%  0.

DIAGNOSTIC

In [9]:
import json, pickle
# ── Evaluation on Test Set (POOLED model, per-symbol metrics) ─────────────────
print("=" * 50)
print("EVALUATION ON TEST SET (pooled model)")
print("=" * 50)

def evaluate(actuals, preds, label):
    mae  = np.mean(np.abs(actuals - preds))
    rmse = np.sqrt(np.mean((actuals - preds) ** 2))
    dir_acc = np.mean(np.sign(actuals) == np.sign(preds)) * 100
    corr = np.corrcoef(actuals, preds)[0, 1]
    print("  [" + label + "]")
    print(f"  MAE {mae:.4f}%  RMSE {rmse:.4f}%  Pearson {corr:.4f}")
    print(f"  Pred std {np.std(preds):.4f}  Actual std {np.std(actuals):.4f}  Dir Acc {dir_acc:.2f}%")

with open("models/pooled_5d/pooled_best_params.json") as f:
    best = json.load(f)["best_params"]
ckpt = torch.load("models/pooled_5d/pooled_best_bilstm.pt", map_location=device, weights_only=True)
h  = ckpt["lstm.weight_ih_l0"].shape[0] // 4
nl = sum(1 for k in ckpt if k.startswith("lstm.weight_ih_l") and "_reverse" not in k)
model = StockPctChangeBiLSTMAttention(input_size=POOLED_INPUT_SIZE, hidden_size=h, num_layers=nl, dropout=best["dropout"]).to(device)
model.load_state_dict(ckpt); model.eval()
mt = best.get("move_threshold", 1.0)

rows = []
for symbol in POOL_SYMS:
    Xte, yte, _ = build_symbol(symbol, "test", best["seq_len"], mt)
    with torch.no_grad():
        pe = model(torch.tensor(Xte, dtype=torch.float32).to(device)).cpu().numpy()
    with open(f"models/{symbol}/{symbol}_target_scaler.pkl", "rb") as f:
        ts = pickle.load(f)
    vol = prepared[symbol]["test_vol_denom"][best["seq_len"]:]
    pe_pct = ts.inverse_transform(pe)[:, 0]  * vol
    ye_pct = ts.inverse_transform(yte)[:, 0] * vol
    print("=" * 55)
    print("  Evaluating -", symbol)
    evaluate(ye_pct, pe_pct, "Price 5d % Change")
    pd.DataFrame({"actual_price_pct": ye_pct, "predicted_price_pct": pe_pct}).to_csv(
        f"models/{symbol}/{symbol}_test_predictions.csv", index=False)
    rows.append((symbol, float(np.corrcoef(ye_pct, pe_pct)[0, 1]),
                 float(np.mean(np.sign(ye_pct) == np.sign(pe_pct)) * 100)))

print("=" * 55)
print("  POOLED MODEL — per-symbol summary")
print("=" * 55)
print(f"{'sym':>6} {'Pearson':>9} {'dirAcc%':>8}")
for s, r, d in rows:
    print(f"{s:>6} {r:>9.4f} {d:>8.2f}")
print(f"{'MEAN':>6} {np.mean([r for _, r, _ in rows]):>9.4f} {np.mean([d for _, _, d in rows]):>8.2f}")
print("  positive-Pearson symbols:", sum(1 for _, r, _ in rows if r > 0), "/", len(rows))


EVALUATION ON TEST SET (pooled model)
  Evaluating - GOOGL
  [Price 5d % Change]
  MAE 4.1324%  RMSE 5.2974%  Pearson -0.0504
  Pred std 3.0882  Actual std 4.1449  Dir Acc 48.24%
  Evaluating - AVGO
  [Price 5d % Change]
  MAE 4.7730%  RMSE 6.6047%  Pearson -0.0030
  Pred std 3.2493  Actual std 5.7395  Dir Acc 51.38%
  Evaluating - MSFT
  [Price 5d % Change]
  MAE 3.2116%  RMSE 4.1118%  Pearson 0.1103
  Pred std 2.2774  Actual std 3.5473  Dir Acc 50.73%
  Evaluating - META
  [Price 5d % Change]
  MAE 4.8014%  RMSE 6.4899%  Pearson 0.0056
  Pred std 3.2239  Actual std 5.6507  Dir Acc 49.49%
  Evaluating - NVDA
  [Price 5d % Change]
  MAE 6.2398%  RMSE 7.9805%  Pearson 0.0718
  Pred std 4.7079  Actual std 6.6882  Dir Acc 50.95%
  Evaluating - AAPL
  [Price 5d % Change]
  MAE 3.8894%  RMSE 5.0667%  Pearson 0.0151
  Pred std 3.1360  Actual std 4.0271  Dir Acc 49.86%
  Evaluating - TSLA
  [Price 5d % Change]
  MAE 8.0748%  RMSE 10.6360%  Pearson -0.0105
  Pred std 5.1196  Actual std 9.2673 

In [10]:
import pandas as pd
import numpy as np

for symbol in symbols:
    csv_path = f"models/{symbol}/{symbol}_test_predictions.csv"

    try:
        results = pd.read_csv(csv_path)
    except FileNotFoundError:
        print(f"\n  Skipping {symbol} — {csv_path} not found.")
        continue

    price_actual    = results["actual_price_pct"]
    price_predicted = results["predicted_price_pct"]

    correct   = (np.sign(price_actual) == np.sign(price_predicted))
    up_mask   = price_actual > 0
    down_mask = price_actual < 0

    print(f"\n{'═'*55}")
    print(f"  {symbol} — Honest Model Baseline")
    print(f"{'═'*55}")
    print(f"  Actual    mean : {price_actual.mean():.4f}   std: {price_actual.std():.4f}")
    print(f"  Predicted mean : {price_predicted.mean():.4f}   std: {price_predicted.std():.4f}")
    print(f"\n  Overall Dir Acc : {correct.mean()*100:.2f}%")
    print(f"  When actual UP  : {correct[up_mask].mean()*100:.2f}%  ({up_mask.sum()} samples)")
    print(f"  When actual DOWN: {correct[down_mask].mean()*100:.2f}%  ({down_mask.sum()} samples)")
    print(f"\n  % predicted UP  : {(price_predicted > 0).mean()*100:.2f}%")
    print(f"  % actual UP     : {(price_actual > 0).mean()*100:.2f}%")

    print(f"\n  Dir Acc by Move Size:")
    bins   = [0, 0.5, 1.0, 2.0, 5.0, 100.0]
    labels = ["<0.5%", "0.5-1%", "1-2%", "2-5%", ">5%"]
    price_actual_abs = price_actual.abs()
    for i, label in enumerate(labels):
        mask = (price_actual_abs >= bins[i]) & (price_actual_abs < bins[i+1])
        if mask.sum() > 0:
            acc = correct[mask].mean() * 100
            print(f"    {label:>8} moves : {acc:.2f}%  ({mask.sum()} samples)")

print(f"\n── Diagnostics done for {len(symbols)} symbols ──")


═══════════════════════════════════════════════════════
  GOOGL — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : 0.6111   std: 4.1460
  Predicted mean : 0.3780   std: 3.0891

  Overall Dir Acc : 48.24%
  When actual UP  : 54.21%  (1070 samples)
  When actual DOWN: 40.05%  (779 samples)

  % predicted UP  : 56.63%
  % actual UP     : 57.87%

  Dir Acc by Move Size:
       <0.5% moves : 44.34%  (212 samples)
      0.5-1% moves : 41.15%  (192 samples)
        1-2% moves : 51.27%  (353 samples)
        2-5% moves : 48.69%  (723 samples)
         >5% moves : 50.41%  (369 samples)

═══════════════════════════════════════════════════════
  AVGO — Honest Model Baseline
═══════════════════════════════════════════════════════
  Actual    mean : 0.9367   std: 5.7410
  Predicted mean : 1.0427   std: 3.2502

  Overall Dir Acc : 51.38%
  When actual UP  : 60.64%  (1039 samples)
  When actual DOWN: 39.51%  (810 samples)

  % predicted UP  : 60.57%
  